# Data sense making

## Introduction and background

In this notebook we perform EDA over a Linked Open Data source, i.e. artresearch.net. **artresearch.net** a federated endpoint aggregating data from 8 major photo archives worldwide
(Frick, Hertziana, KHI, Marburg, PMC, RKD, Warburg, Zeri). The data are organised according to the **CIDOC-CRM** ontology and include detailed information about artworks and photographs of the former. For each artefact is recorded a wealth of information about creators, keepers, subjects, type of objects etc..


In particular we go through the 7 stages of EDA wrt two case studies:

 * statistical analysis - "What are the most recurring artists across data providers of artresearch?"
 * temporal analysis -

## 1. EDA

Usually, the first step in the analysis of a data source is to understand the added value of a given dataset. This often requires the exploration to be performed over the whole dataset or part of it (filtered).

In our running example, we explore artresearch to retrieve some figures that help us to understand the scope and the representativeness of the dataset, so that we can make informed decisions on what kind of analyses are worthed doing on it (e.g. if Chinese artworks are not well represented in the dataset, we cannot use it to explore the changes in Chinese iconography over time).

### 1.1 Understand the data model

Here you find a brief **documentation** on the ontology patterns used in the artresearch dataset.

[https://gist.github.com/aindlq/e37944f24b6ea349ad61bd326170826b](https://gist.github.com/aindlq/e37944f24b6ea349ad61bd326170826b)

Alternatively, we can perform **exploratory queries** over the SPARQL endpoint to retrieve the most used classes and properties

 * [Most common classes](https://artresearch.net/sparql#query=SELECT+%3Fclass+(COUNT(%3Fs)+AS+%3Fcount)%0AWHERE+%7B%0A++%3Fs+a+%3Fclass+.%0A%7D%0AGROUP+BY+%3Fclass%0AORDER+BY+DESC(%3Fcount)%0ALIMIT+50)
 * [Most common properties](https://artresearch.net/sparql#query=%0ASELECT+%3Fproperty+(COUNT(*)+AS+%3Fcount)%0AWHERE+%7B%0A++%3Fs+%3Fproperty+%3Fo+.%0A%7D%0AGROUP+BY+%3Fproperty%0AORDER+BY+DESC(%3Fcount)%0ALIMIT+50)

We could perform more sophisticated queries to appreciate the most common triple patterns, like the following ones. However, these queries are very expensive for a SPARQL endpoint, and are likely to fail or get a timeout error.

Most common triple patterns (s-type, property, o-type)

```

SELECT ?subjectClass ?property ?objectClass (COUNT(*) AS ?count)
WHERE {
  ?s ?property ?o .
  OPTIONAL { ?s a ?subjectClass }
  OPTIONAL { ?o a ?objectClass }
}
GROUP BY ?subjectClass ?property ?objectClass
ORDER BY DESC(?count)
LIMIT 100
```

Most common triple patterns (s-type, property, literal vs uri)

```
SELECT ?subjectClass ?property ?objectType (COUNT(*) AS ?count)
WHERE {
  ?s ?property ?o .
  ?s a ?subjectClass .
  BIND(IF(isLiteral(?o), "literal", IF(isIRI(?o), "iri", "bnode")) AS ?objectType)
}
GROUP BY ?subjectClass ?property ?objectType
ORDER BY DESC(?count)
LIMIT 100
```


We can **identify some rich example** (e.g. see [Mona Lisa record](https://artresearch.net/resource/pharos/artwork/0c6efbedc68a28df2a0fc93479202c6533b8dcd4))  and perform a `DESCRIBE` query tro retrieve common triple patterns (although it might not be exhaustive).

```
DESCRIBE <https://artresearch.net/resource/pharos/artwork/0c6efbedc68a28df2a0fc93479202c6533b8dcd4>
```

## 1.2 Query the dataset programmatically

Ideally, the EDA runs in a notebook (local or remote, like Colab) and programmatically queries a data source, which can be local or remote as well.

To programmatically query a SPARQL endpoint there are several (equally valid) methods and libraries. We will see a few ones.


In [47]:
# NO DEPENDENCIES
import requests

sparql_endpoint = "https://artresearch.net/sparql"
query = "DESCRIBE <https://artresearch.net/resource/pharos/artwork/0c6efbedc68a28df2a0fc93479202c6533b8dcd4>"

headers = {
    "Accept": "text/turtle"  # Requesting Turtle
}

params = {
    "query": query
}

try:
    response = requests.post(sparql_endpoint, data=params, headers=headers)
    response.raise_for_status() # Raise an HTTPError
    print(response.text)
except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")


@prefix assets: <https://artresearch.net/resource/assets> .
@prefix ql: <http://qlever.cs.uni-freiburg.de/builtin-functions/> .
@prefix fedsail: <http://www.openrdf.org/config/sail/federation#> .
@prefix crmtex: <http://www.cidoc-crm.org/extensions/crmtex/> .
@prefix crmarchaeo: <http://www.cidoc-crm.org/extensions/crmarchaeo/> .
@prefix Help: <http://help.researchspace.org/resource/> .
@prefix bds: <http://www.bigdata.com/rdf/search#> .
@prefix custom: <https://artresearch.net/custom/> .
@prefix fc: <https://artresearch.net/resource/fc/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix crmba: <http://www.cidoc-crm.org/extensions/crmba/> .
@prefix pharos: <https://artresearch.net/resource/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix crm: <http://www.cidoc-crm.org/cidoc-crm/> .
@prefix textSearch: <https://qlever.cs.uni-freiburg.de/textSearch/> .
@prefix crmgeo: <http://www.cidoc-crm.org/extensions/crmgeo/> 

# Statistical analysis

Statistical analysis includes a number of algorithms, rules, and models that can be used to understand phenomena underlying the surface of data.

In EDA, statistics are meant to frame the scope of the dataset and define whether the source is complete or representative enough to perform some further analysis.

For instance, we answer the following question: `"What are the most recurring artists across data providers of artresearch?"` in order to estimate which artists can be considered the most interesting by cultural institutions and their patrons.

The question helps us to frame the subset of artists and artworks that can undergo further analyses - for instance, to answer more sophisticated questions such as `which archives are likely to include similar materials? which artistic periods are mostly studied by patrons of such photo archives?`.

**In EDA, the answer to a quantitative question is ONLY the premise to a bigger, more meaningful question. It's the beginning of the analysis, not the end.**

To answer the question, we go through the 7 stages of EDA:

- acquire (from SPARQL endpoint via SPARQL query)
- parse (with appropriate libraries for the data structure at hand)
- filter (use the triple patterns that frame the expected results)
- mine (transform results in tables / data frames)
- represent (use appropriate libraries and charts for data visualisation)
- refine (highlight meaningful results in title, annotations etc)
- interact (let the user interact with the chart)





## 💡 What are the most recurring artists across data providers of artresearch?

**Acquire**

Run the query programmatically over the SPARQL endpoint as shown below.

**Parse**

We retrieve data from the SPARQL endpoint API and we specify `application/sparql-results+json` as format. This format require us to parse results as a JSON document, for instance using methods of the library `requests`.

**Filter**

The query below applies a filter on artists documented in more than one dataset.

**Mine**

We manipulate the returned JSON response to transform it into a dataframe, a format that is likely to fit the requirements of any charting library. To do so, we need to transform `k:v` pairs into a list of lists (each list corresponding to a variable of the `k:v` pairs and later to a column of the dataframe).

In [48]:
from IPython.display import clear_output
clear_output()

import requests
import pandas as pd
import time

sparql_endpoint = "https://artresearch.net/sparql"
query = """
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX pharos-meta: <https://artresearch.net/resource/pharos/vocab/meta/>

SELECT ?artist SAMPLE(?aname AS ?name) (COUNT(DISTINCT ?dataset) AS ?count) (GROUP_CONCAT(DISTINCT STR(?dataset); separator=" | ") AS ?institutions)
WHERE {
  ?work crm:P108i_was_produced_by ?production ; crm:P70i_is_documented_in ?dataset .
  ?production crm:P14_carried_out_by ?artist .
  FILTER(STRSTARTS(STR(?dataset), "https://artresearch.net/resource/e31/"))
  OPTIONAL {
    ?artist crm:P1_is_identified_by ?app .
    ?app crm:P2_has_type/crm:P127_has_broader_term* pharos-meta:preferred_name ;
         crm:P190_has_symbolic_content ?aname .
  }
}
GROUP BY ?artist ?name
HAVING (COUNT(DISTINCT ?dataset) > 1)
ORDER BY DESC(?count)
"""


headers = {
    "Accept": "application/sparql-results+json",
    "Cache-Control": "no-cache, no-store",
    "Pragma": "no-cache"
}

params = {
    "query": query+"&nocache=1",
    "cache_bust": time.time()  # forces a fresh request every time
}

try:
    response = requests.post(sparql_endpoint, data=params, headers=headers)
    response.raise_for_status() # Raise an HTTPError

    json_data = response.json()

    # Extract variables from the SPARQL query result
    variables = json_data['head']['vars']

    # Extract bindings (rows of data)
    results = []
    for binding in json_data['results']['bindings']:
        row = {var: binding[var]['value'] for var in variables}
        results.append(row)

    # Create a Pandas DataFrame
    df = pd.DataFrame(results)
    display(df) # show first 50 rows

except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")
except ValueError as e:
    print(f"Error decoding JSON response: {e}")
except KeyError as e:
    print(f"Unexpected JSON structure: Missing key {e}")


,artist,count
0,http://vocab.getty.edu/ulan/500010879,6
1,http://vocab.getty.edu/ulan/500003965,5
2,http://vocab.getty.edu/ulan/500029319,5
3,http://viaf.org/viaf/250803549,4
4,http://vocab.getty.edu/ulan/500017301,4
5,http://vocab.getty.edu/ulan/500011558,4
6,http://vocab.getty.edu/ulan/500014786,4
7,http://vocab.getty.edu/ulan/500028698,4
8,http://vocab.getty.edu/ulan/500000246,4
9,http://vocab.getty.edu/ulan/500021065,4


**NB.** *The previous query may return incorrect results due to cache mechanisms in place in the SPARQL endpoint. The query runs correctly on the endpoint GUI and can be exported as JSON. In the following charts we use as input a copy of results stored on the github repository of the course instead of the live results of the above query.*

**Respresent**

Based on the question, we can choose the appropriate chart, depending on the answer we expect.

1. To highlight *which* specific artists are the most recurring across datasets, we could use a bar chart - good for bivariate analysis, it shows the distribution of a categorical value (the artist) in a dataset. If sorted in descending order, you immediately appreciate the most recurring artists' names.

2. To appreciate how many artists are shared across institutions on average, we could use another bar chart - that tells us how many artists fit under a certain class (or bin), being the class the number of archives. In other terms, it helps us to answer questions like `how many artists are shared across 7 / 6 / 5 / 4 / 3 / ... institutions?`. this way we understand if there is a large subset of popular artists (those appearing in all archives) or just a few.

3. Some archives may share a bigger subsets of artists than others. A sorted bar chart, again, can help us to appreciate the archive that has the highest number of artists that appear in other archives too.

4. Some archives may share similar subsets of artists. A co-occurrence heatmap can help us to appreciate which archives co-occur the most in the dataset, so that we can estimate where to find most similar materials.

**refine**

In the first bar chart, since the returned results are many to fit in a chart, we refine results to show only the first 100 most represented artists. To highlight groups, we color bars according to the number of archives, using shades of blue (darker = higher number).

**interact**

We use plotly, a charting library that works well in Colab / Jupyter to plot a bar chart. Each bar represents an artist, and the value represents the number of archives that share artwork records of that artist.

### Most recurring artists across archives

In [49]:
import requests
import pandas as pd
import plotly.express as px

json_url = "https://raw.githubusercontent.com/marilenadaquino/information_visualization/main/2025-2026/tutorials/artists_by_archive.json"

response = requests.get(json_url)
json_data = response.json()

variables = json_data['head']['vars']
results = []
for binding in json_data['results']['bindings']:
    row = {}
    for var in variables:
        # Safely get the value for each variable, handling cases where a key might be missing
        row[var] = binding.get(var, {}).get('value', '') # Provide empty string if key or value is missing
    results.append(row)

df = pd.DataFrame(results)
df['count'] = pd.to_numeric(df['count'])
df = df.sort_values(by='count', ascending=False).head(100)  # top 100

# extract readable label from URI, use 'name' if available, otherwise default to artist URI part
df['artist_label'] = df['name'].apply(lambda x: x if x else df['artist'].str.extract(r'/([^/]+)$'))

# Fallback for artist_label if name is empty and URI part extraction fails (unlikely, but good practice)
df['artist_label'] = df.apply(lambda row: row['name'] if row['name'] else str(row['artist']).split('/')[-1], axis=1)

fig = px.bar(
    df,
    x='count',
    y='artist_label',
    orientation='h',  # horizontal is much more readable for many labels
    title='Most Recurring Artists Across Archives',
    labels={'artist_label': 'Artist', 'count': 'Number of Datasets'},
    color='count',
    color_continuous_scale='Blues',
    hover_data={'artist': True}  # show full URI on hover
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=800,
    coloraxis_showscale=False
)

fig.show()


The chart above allows a suer to interact with it and discover the names of artists that are described by archives in artresearch. It appears that most shared artists are indeed popular artists, mostly from modern age.

### Distribution of artists by number of artists sharing artist records

In [50]:
import requests
import pandas as pd
import plotly.express as px

# URL of the JSON file
json_url = "https://raw.githubusercontent.com/marilenadaquino/information_visualization/main/2025-2026/tutorials/artists_by_archive.json"

try:
    # Fetch the JSON data
    response = requests.get(json_url)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    json_data = response.json()

    # Extract variables from the SPARQL query result
    variables = json_data['head']['vars']

    # Extract bindings (rows of data)
    results = []
    for binding in json_data['results']['bindings']:
        row = {}
        for var in variables:
            # Safely get the value for each variable, handling cases where a key might be missing
            row[var] = binding.get(var, {}).get('value', '') # Provide empty string if key or value is missing
        results.append(row)

    # Create a Pandas DataFrame
    df_new = pd.DataFrame(results)

    # Ensure 'count' column is numeric for plotting
    df_new['count'] = pd.to_numeric(df_new['count'])

    # Display the new DataFrame head to confirm data
    print("DataFrame loaded from JSON:")
    display(df_new.head())

    # Create the histogram
    fig = px.histogram(
        df_new,
        x='count',
        title='Distribution of Artists by Number of Shared Archives',
        labels={'count': 'Number of Archives', 'count': 'Number of Artists'},
        nbins=len(df_new['count'].unique()), # Set number of bins based on unique archive counts
        color_discrete_sequence=['darkblue'] # Optional: set a single color
    )

    fig.update_layout(
        xaxis_title_text='Number of Archives',
        yaxis_title_text='Number of Artists',
        bargap=0.05 # Gap between bars for better readability
    )

    fig.show()

except requests.exceptions.RequestException as e:
    print(f"An error occurred while fetching data: {e}")
except ValueError as e:
    print(f"Error decoding JSON response: {e}")
except KeyError as e:
    print(f"Unexpected JSON structure: Missing key {e}")


DataFrame loaded from JSON:


,artist,name,count,institutions
0,http://vocab.getty.edu/ulan/500005125,"Trevisani, Francesco",7,https://artresearch.net/resource/e31/frick | h...
1,http://vocab.getty.edu/ulan/500010879,Leonardo da Vinci,7,https://artresearch.net/resource/e31/frick | h...
2,http://vocab.getty.edu/ulan/500028857,"Ricci, Sebastiano",7,https://artresearch.net/resource/e31/frick | h...
3,http://vocab.getty.edu/ulan/500115339,"Canova, Antonio",7,https://artresearch.net/resource/e31/marburg |...
4,http://vocab.getty.edu/ulan/500115312,Caravaggio,7,https://artresearch.net/resource/e31/frick | h...


The chart reveals that the overlap across archives in artresearch is actually relatively small. Only a few artists seem to appear across all or almost all archives, while the majority of artists mostly appear in 2-3 archives.

### Number of artists shared by each archive

In [51]:
import pandas as pd
import plotly.express as px
import requests

json_url = "https://raw.githubusercontent.com/marilenadaquino/information_visualization/main/2025-2026/tutorials/artists_by_archive.json"

try:
    response = requests.get(json_url)
    response.raise_for_status()
    json_data = response.json()

    variables = json_data['head']['vars']
    results = []
    for binding in json_data['results']['bindings']:
        row = {}
        for var in variables:
            row[var] = binding.get(var, {}).get('value', '')
        results.append(row)

    df_full = pd.DataFrame(results)

    # Split the 'institutions' column by '|' and explode to get one row per institution per artist
    df_institutions = df_full['institutions'].str.split(' | ', expand=True).stack().reset_index(level=1, drop=True).to_frame('institution')

    # Count the number of unique artists per institution
    # Note: df_full contains an 'artist' column which is the URI of the artist.
    # We need to join this back to count unique artists per institution.
    df_temp = df_full[['artist', 'institutions']]
    df_exploded = df_temp.assign(institution=df_temp['institutions'].str.split(' | ')).explode('institution')

    # Filter out empty institution strings AND the '|' character itself
    df_exploded = df_exploded[df_exploded['institution'] != '']
    df_exploded = df_exploded[df_exploded['institution'] != '|']

    # Group by institution and count unique artists
    institution_artist_counts = df_exploded.groupby('institution')['artist'].nunique().reset_index()
    institution_artist_counts.columns = ['institution', 'unique_artists_count']

    # Sort for better visualization
    institution_artist_counts = institution_artist_counts.sort_values(by='unique_artists_count', ascending=False)

    # Extract a more readable label for institutions from the URI
    institution_artist_counts['institution_label'] = institution_artist_counts['institution'].apply(lambda x: x.split('/')[-1])

    print("Institutions with their unique artist counts:")
    display(institution_artist_counts.head())

    # Create the bar chart
    fig = px.bar(
        institution_artist_counts,
        x='institution_label',
        y='unique_artists_count',
        title='Number of Unique Artists Shared by Each Archive',
        labels={'institution_label': 'Archive', 'unique_artists_count': 'Number of Unique Artists'},
        color='unique_artists_count',
        color_continuous_scale='Viridis'
    )

    fig.update_layout(
        xaxis_title_text='Archive',
        yaxis_title_text='Number of Unique Artists',
        xaxis_tickangle=-45 # Rotate labels if they are too long
    )

    fig.show()

except requests.exceptions.RequestException as e:
    print(f"An error occurred while fetching data: {e}")
except ValueError as e:
    print(f"Error decoding JSON response: {e}")
except KeyError as e:
    print(f"Unexpected JSON structure: Missing key {e}")

Institutions with their unique artist counts:


,institution,unique_artists_count,institution_label
0,https://artresearch.net/resource/e31/frick,3300,frick
6,https://artresearch.net/resource/e31/warburg,2146,warburg
7,https://artresearch.net/resource/e31/zeri,2130,zeri
4,https://artresearch.net/resource/e31/pmc,1449,pmc
3,https://artresearch.net/resource/e31/marburg,730,marburg


The chart shows that the Frick Art Reference Library has by far the highest number of artists that also appear in other archives. That is, is the archive that might benefit the most by data integration, since a high number of their records may also appear in other archives, completing or updating their information. Conversely, institutions like Hertziana share little artists (and records) with other institutions, which makes the institution rather unique in terms of scope and content.

**NB.** *The absence of artists in RKD is due to the incomplete reconciliation of artists to ULAN.*

### Co-occurrence of artists in archives

In [52]:
import pandas as pd
import plotly.express as px
import requests

# URL of the JSON file (re-using the same URL as before)
json_url = "https://raw.githubusercontent.com/marilenadaquino/information_visualization/main/2025-2026/tutorials/artists_by_archive.json"

try:
    # Fetch the JSON data
    response = requests.get(json_url)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    json_data = response.json()

    # Extract variables and bindings as done previously
    variables = json_data['head']['vars']
    results = []
    for binding in json_data['results']['bindings']:
        row = {}
        for var in variables:
            row[var] = binding.get(var, {}).get('value', '')
        results.append(row)

    df_full = pd.DataFrame(results)

    # Ensure 'count' column is numeric
    df_full['count'] = pd.to_numeric(df_full['count'])

    # Prepare data for co-occurrence matrix
    # Explode the 'institutions' column to get a row for each artist-institution pair
    df_exploded = df_full.assign(institution=df_full['institutions'].str.split(' | ')).explode('institution')

    # Filter out empty institution strings AND the '|' character itself
    df_exploded = df_exploded[df_exploded['institution'] != '']
    df_exploded = df_exploded[df_exploded['institution'] != '|']

    # Create a binary matrix where rows are artists and columns are institutions
    # Value is 1 if the artist is associated with the institution, 0 otherwise
    artist_institution_matrix = df_exploded.pivot_table(
        index='artist',
        columns='institution',
        aggfunc='size',
        fill_value=0
    )

    # Calculate the co-occurrence matrix
    # This matrix will show how many artists are shared between each pair of institutions
    cooccurrence_matrix = artist_institution_matrix.T.dot(artist_institution_matrix)

    # Replace institution URIs with more readable labels
    institution_labels = {uri: uri.split('/')[-1] for uri in cooccurrence_matrix.columns}
    cooccurrence_matrix = cooccurrence_matrix.rename(columns=institution_labels, index=institution_labels)

    # Display the co-occurrence matrix head
    print("Co-occurrence Matrix (first 5x5):")
    display(cooccurrence_matrix.head())

    # Create the heatmap
    fig = px.imshow(
        cooccurrence_matrix,
        text_auto=True, # Show values on the heatmap cells
        color_continuous_scale='Viridis',
        title='Shared Artists Between Archives (Co-occurrence Heatmap)',
        labels=dict(x="Archive 1", y="Archive 2", color="Number of Shared Artists")
    )

    fig.update_xaxes(side="top") # Move x-axis labels to the top for better readability
    fig.update_layout(height=800, width=900) # Adjust figure size if needed

    fig.show()

except requests.exceptions.RequestException as e:
    print(f"An error occurred while fetching data: {e}")
except ValueError as e:
    print(f"Error decoding JSON response: {e}")
except KeyError as e:
    print(f"Unexpected JSON structure: Missing key {e}")


Co-occurrence Matrix (first 5x5):


institution,frick,hertziana,khi,marburg,pmc,rkd,warburg,zeri
institution,,,,,,,,
frick,3300,291,279,348,1322,1,1675,1124
hertziana,291,445,231,244,22,1,182,345
khi,279,231,568,231,9,1,180,517
marburg,348,244,231,730,27,1,292,709
pmc,1322,22,9,27,1449,0,411,99


Not surprisingly, American and British institutions (Frick, Warburg, PMC) share the most artists, therefore being the most similar in terms of scope of their archives.

German institutions (Hertziana, Marburg, KHI) mostly share artists with the Zeri foundation, an Italian institutions with which they share a longstanding collaboration.

The Zeri foundation, in turn, shares the most artists with Frick and Warburg, which highlights preferences and networks of its funder (Federico Zeri).

# Temporal analysis

The previous analysis tells us which artists are mostly shared across archives. However, it is not immediate to understand which historical periods they do cover, and therefore which artistic periods overlap across archives.

## 💡 What are the most represented periods in data providers of artresearch?

Temporal analysis is the study of how data distributes over time. It allows us to ask questions like: `which historical periods are best documented? Which institutions focus on medieval art vs. modern art? Where are the gaps in coverage?`

We query artresearch.net to retrieve each artwork in
the graph that has a production time-span modeled using CIDOC-CRM properties
`P82a_begin_of_the_begin` and `P82b_end_of_the_end` — the earliest possible start and latest possible end of the production period.

By extracting these dates and aggregating them by century, we can visualize
the temporal coverage of records belonging to each institution and compare their collection profiles:
- Does the Frick focus on different centuries than the Zeri?
- Which centuries are underrepresented across all archives?
- Are there institutions that complement each other temporally?

This is a simple but powerful form of exploratory data analysis (EDA) on
semantic data: instead of querying a relational database, we traverse a
knowledge graph using SPARQL, then use Python (pandas + plotly) to
analyze and visualize the results.

We go through the 7 stages again, by acquiring and parsing data from the SPARQL endpoint of artresearch. We transform results into a dataframe and we select a multi-bar chart for visualising time spans - it allows for precise comparison and interactive pruning of series.

In [65]:
import requests
import pandas as pd
import plotly.express as px
import time

sparql_endpoint = "https://artresearch.net/sparql"
headers = {"Accept": "application/sparql-results+json", "Cache-Control": "no-cache", "Pragma": "no-cache"}

institutions = ["frick", "hertziana", "khi", "marburg", "pmc", "rkd", "warburg", "zeri"]

def fetch(inst):
    query = f"""
    PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
    SELECT ?work ?start ?end
    WHERE {{
      ?work crm:P108i_was_produced_by ?production ;
            crm:P70i_is_documented_in <https://artresearch.net/resource/e31/{inst}> .
      ?production crm:P4_has_time-span ?ts .
      ?ts crm:P82a_begin_of_the_begin ?start .
      OPTIONAL {{ ?ts crm:P82b_end_of_the_end ?end . }}
    }}

    """
    # use GET instead of POST, with query as URL param
    r = requests.get(
        sparql_endpoint,
        params={"query": query, "_": int(time.time())},  # _ param forces unique URL
        headers=headers
    )
    print(f"{inst}: status={r.status_code}, size={len(r.content)}")
    if r.status_code != 200:
        return []
    bindings = r.json()['results']['bindings']
    print(f"{inst}: {len(bindings)} bindings — sample: {bindings[0] if bindings else 'empty'}")
    return [{
        "institution": inst,
        "start": b["start"]["value"] if "start" in b else None,
        "end": b["end"]["value"] if "end" in b else None
    } for b in bindings]

rows = []
for inst in institutions:
    rows.extend(fetch(inst))

df = pd.DataFrame(rows)
df = df.dropna(subset=['start'])
df['year'] = df['start'].str[:4].astype(int)
df['century'] = (df['year'] // 100) * 100

pivot = df.groupby(['institution', 'century']).size().reset_index(name='count')
print(pivot)

fig = px.bar(
    pivot,
    x='century',
    y='count',
    color='institution',
    barmode='group',
    title='Temporal Coverage by Institution (sample of 50 per institution)',
    labels={'century': 'Century', 'count': 'Works'}
)
fig.show()

frick: status=200, size=82858403
frick: 257969 bindings — sample: {'work': {'type': 'uri', 'value': 'https://artresearch.net/resource/frick/work/991000003979707141'}, 'start': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '1801-01-01T00:00:00Z'}, 'end': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '1850-12-31T23:59:59Z'}}
hertziana: status=200, size=760395
hertziana: 2191 bindings — sample: {'work': {'type': 'uri', 'value': 'https://artresearch.net/resource/pharos/artwork/337999ea66406124891c99238e55b1860e59b8f9'}, 'start': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '1500-01-01T00:00:00Z'}, 'end': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '1599-12-31T23:59:59Z'}}
khi: status=200, size=596264
khi: 1718 bindings — sample: {'work': {'type': 'uri', 'value': 'https://artresearch.net/resource/pharos/artwork/4ab9f7502add479ca25

# Exercise

## 💡 Which artists are most represented in each archive?

Write a SPARQL query that retrieves, for each institution, the top 10 artists by number of works. Then visualize the results as a horizontal bar chart, one panel per institution.

Start with this exploratory query that returns the 10 most productive artists at the Frick Art Reference Library.

```
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>

SELECT ?artist (COUNT(?work) AS ?count)
WHERE {
  ?work crm:P108i_was_produced_by ?production ;
        crm:P70i_is_documented_in <https://artresearch.net/resource/e31/frick> .
  ?production crm:P14_carried_out_by ?artist .
}
GROUP BY ?artist
ORDER BY DESC(?count)
LIMIT 10
```

Tasks:

1. Run the query for one institution and plot the results
2. Modify it to also retrieve the artist's preferred name (add the appellation pattern)
3. Loop over all institutions and produce a subplot for each